# LLM-Powered Data Pipeline

This notebook demonstrates chaining LLM steps to process text data at scale:
classify sentiment, extract entities, and summarize — outputting structured results
as a pandas DataFrame.

## Provider Configuration

In [ ]:
import pandas as pd
from dotenv import load_dotenv
from langchain_anthropic import ChatAnthropic
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field

# from langchain_openai import ChatOpenAI
# from langchain_google_genai import ChatGoogleGenerativeAI
# from langchain_ollama import ChatOllama

load_dotenv()

# Provider configuration — uncomment your preferred provider

llm = ChatAnthropic(model="claude-sonnet-4-20250514")

# llm = ChatOpenAI(model="gpt-4o")

# llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

# llm = ChatOllama(model="llama3.2")

## Sample Data: Product Reviews

Synthetic reviews for demonstrating the pipeline. In production, this would
come from a database, CSV, or API.

In [ ]:
reviews = [
    {
        "id": 1,
        "text": "Absolutely love this coffee maker! Brews fast and the temperature is perfect every time. Best purchase this year.",
        "rating": 5,
    },
    {
        "id": 2,
        "text": "The blender stopped working after two weeks. Motor burned out and customer service was unhelpful. Total waste of money.",
        "rating": 1,
    },
    {
        "id": 3,
        "text": "Decent toaster for the price. Toast comes out evenly but the timer is a bit unreliable. Gets the job done.",
        "rating": 3,
    },
    {
        "id": 4,
        "text": "This air fryer changed how I cook. Everything comes out crispy with barely any oil. Easy to clean too.",
        "rating": 5,
    },
    {
        "id": 5,
        "text": "Microwave works fine but the door handle feels cheap and the beeping is extremely loud. Annoying but functional.",
        "rating": 3,
    },
    {
        "id": 6,
        "text": "Returned immediately. The electric kettle leaked from day one. Dangerous product that should be recalled.",
        "rating": 1,
    },
    {
        "id": 7,
        "text": "Solid stand mixer. Heavy and sturdy, handles thick dough without struggling. A bit noisy on high speed though.",
        "rating": 4,
    },
    {
        "id": 8,
        "text": "The rice cooker makes perfect rice every time. Set it and forget it. Wish it had a larger capacity though.",
        "rating": 4,
    },
    {
        "id": 9,
        "text": "Dishwasher leaves spots on glasses and doesn't dry properly. For this price I expected much better performance.",
        "rating": 2,
    },
    {
        "id": 10,
        "text": "Incredible espresso machine. Pulls shots like a cafe. The learning curve is steep but worth every penny.",
        "rating": 5,
    },
]

## Define Structured Output Schema

Using Pydantic models to ensure the LLM returns structured, typed data.

In [ ]:
class ReviewAnalysis(BaseModel):
    """Structured analysis of a single product review."""

    sentiment: str = Field(description="One of: positive, negative, neutral")
    product_features: list[str] = Field(
        description="Key product features or aspects mentioned in the review"
    )
    summary: str = Field(description="One-sentence summary of the review")

## Build the LCEL Pipeline

LangChain Expression Language (LCEL) lets us compose prompt → LLM → parser
into a single chain using the `|` (pipe) operator.

In [ ]:
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a product review analyst. Analyze the given review and extract "
            "structured information. Be concise and precise.",
        ),
        (
            "human",
            "Analyze this product review:\n\n{review_text}",
        ),
    ]
)

chain = prompt | llm.with_structured_output(ReviewAnalysis)

## Process a Single Review

Test the chain on one review before running the full batch.

In [ ]:
result = chain.invoke({"review_text": reviews[0]["text"]})
print(f"Sentiment: {result.sentiment}")
print(f"Features: {result.product_features}")
print(f"Summary: {result.summary}")

## Batch Process All Reviews

Use `chain.batch()` to process all reviews efficiently.

In [ ]:
inputs = [{"review_text": r["text"]} for r in reviews]
results = chain.batch(inputs)

print(f"Processed {len(results)} reviews")

## Results as a DataFrame

Combine the structured LLM output with the original review data.

In [ ]:
analysis_df = pd.DataFrame(
    {
        "id": [r["id"] for r in reviews],
        "rating": [r["rating"] for r in reviews],
        "sentiment": [r.sentiment for r in results],
        "product_features": [", ".join(r.product_features) for r in results],
        "summary": [r.summary for r in results],
    }
)

analysis_df

In [ ]:
print("Sentiment distribution:")
print(analysis_df["sentiment"].value_counts().to_string())
print("\nAverage rating by sentiment:")
print(analysis_df.groupby("sentiment")["rating"].mean().to_string())

## Notes

- `with_structured_output()` uses the LLM's native tool/function calling — no regex parsing
- `chain.batch()` processes inputs concurrently for better throughput
- Pydantic validation ensures type safety — malformed LLM output raises clear errors
- This pattern scales to thousands of records with `chain.batch(inputs, config={"max_concurrency": 5})`
- Works with any provider that supports structured output (OpenAI, Anthropic, Google, newer Ollama models)